In [18]:
%load_ext autoreload
%autoreload 2

In [7]:
import asyncio
from pathlib import Path
import pandas as pd
from labellers.label_transcript import label_transcript_with_topics
COMPANY = "MU"
TRANSCRIPT_DIR = Path(f"data/Transcripts/{COMPANY}")
TOPIC_CSV_DIR = Path(f"data/topic_dataset/{COMPANY}")
COOLDOWN_SECONDS = 60

In [8]:
# Change only this value to process a different company.
transcript_files = sorted(TRANSCRIPT_DIR.glob("*.txt"))

print(f"Found {len(transcript_files)} {COMPANY} transcripts")
for idx, transcript_file in enumerate(transcript_files):
    existing_csv = TOPIC_CSV_DIR / f"{transcript_file.stem}.csv"
    if existing_csv.exists():
        print(f"Skipping {transcript_file.name} (already exists: {existing_csv.name})")
        continue

    print(f"Processing {transcript_file.name}")
    await label_transcript_with_topics(str(transcript_file))
    if idx < len(transcript_files) - 1:
        print(f"Cooldown: waiting {COOLDOWN_SECONDS} seconds before next transcript...")
        await asyncio.sleep(COOLDOWN_SECONDS)

print(f"Done processing {COMPANY} transcripts")


Found 17 MU transcripts
Processing 2016-Dec-21-MU.txt
Labelling sentences...


Labelling sentences: 100%|██████████| 30/30 [00:05<00:00,  5.57batch/s]


Cooldown: waiting 60 seconds before next transcript...
Processing 2016-Jun-30-MU.txt
Labelling sentences...


Labelling sentences: 100%|██████████| 29/29 [00:03<00:00,  7.54batch/s]


Cooldown: waiting 60 seconds before next transcript...
Processing 2016-Mar-30-MU.txt
Labelling sentences...


...

Cooldown: waiting 60 seconds before next transcript...
Processing 2020-Jun-29-MU.txt
Labelling sentences...


Labelling sentences: 100%|██████████| 37/37 [00:10<00:00,  3.69batch/s]


Cooldown: waiting 60 seconds before next transcript...
Processing 2020-Mar-25-MU.txt
Labelling sentences...


Labelling sentences: 100%|██████████| 37/37 [00:06<00:00,  5.52batch/s]

Done processing MU transcripts


In [9]:
csv_files = sorted(TOPIC_CSV_DIR.glob("*.csv"))
invalid_count = 0
if not csv_files:
    print(f"No CSV files found for {COMPANY} in {TOPIC_CSV_DIR}")
else:
    issues_found = False

    for csv_path in csv_files:
        print(csv_path)
        df = pd.read_csv(csv_path)

        if "topic" not in df.columns:
            issues_found = True
            print(f"\n[Missing column] {csv_path.name} has no 'topic' column.")
            continue

        # Convert to numeric; non-numeric values become NaN
        numeric_col = pd.to_numeric(df["topic"], errors="coerce")

        # Valid values are 1-10 for topic classification
        invalid_mask = numeric_col.isna() | ~numeric_col.isin(range(1, 12))

        if invalid_mask.any():
            issues_found = True
            bad_rows = df.loc[invalid_mask, ["sentence", "topic"]].copy()
            print(f"\n[Invalid values] {csv_path.name}")
            print(bad_rows.to_string(index=False))
            invalid_count += 1

    if not issues_found:
        print(f"All {COMPANY} files passed: every 'topic' value is numeric 1-11.")
    else:
        print(f"\n{invalid_count} files have invalid 'topic' values. Please review the above details.")

data\topic_dataset\MU\2016-Dec-21-MU.csv
data\topic_dataset\MU\2016-Jun-30-MU.csv
data\topic_dataset\MU\2016-Mar-30-MU.csv
data\topic_dataset\MU\2016-Oct-04-MU.csv
data\topic_dataset\MU\2017-Dec-19-MU.csv
data\topic_dataset\MU\2017-Jun-29-MU.csv
data\topic_dataset\MU\2017-Mar-23-MU.csv
data\topic_dataset\MU\2017-Sep-26-MU.csv
data\topic_dataset\MU\2018-Dec-18-MU.csv
data\topic_dataset\MU\2018-Mar-22-MU.csv
data\topic_dataset\MU\2018-Sep-20-MU.csv
data\topic_dataset\MU\2019-Dec-18-MU.csv
data\topic_dataset\MU\2019-Jun-25-MU.csv
data\topic_dataset\MU\2019-Mar-20-MU.csv
data\topic_dataset\MU\2019-Sep-26-MU.csv
data\topic_dataset\MU\2020-Jun-29-MU.csv
data\topic_dataset\MU\2020-Mar-25-MU.csv
All MU files passed: every 'topic' value is numeric 1-11.
